# CogFlex Notebook Overview

This notebook defines and registers the Kaggle runtime for `cogflex_suite_flexible`. It is intended to be executed from top to bottom.

## Flow

1. Initialize imports, constants, dataset paths, and split selection.
2. Define the runtime error used during prompt execution and scoring.
3. Normalize model outputs into the ordered-label format expected by the benchmark.
4. Load and validate benchmark rows, and attach private scoring targets when needed.
5. Execute each episode against the model and convert predictions into scores.
6. Summarize results and register the final Kaggle task.
7. Run the main benchmark task once to materialize its latest run file.
8. Activate the task with `%choose cogflex_suite_flexible`.

## Split behavior

- **Public split**: rows already contain the scoring targets needed for evaluation.
- **Private split**: rows remain inference-only, and hidden targets are joined from a separate answer key during preparation.

## What this notebook provides

By the end of the notebook, Kaggle has a fully registered task that can load the selected split, run the benchmark episodes, score model outputs, and report aggregate metrics.

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

import json
import re
import sys
from dataclasses import is_dataclass
from pathlib import Path

# Registered task name used by Kaggle Benchmarks.
TASK_NAME = "CogFlex Suite Task"

# Faculty identifier used to categorize the benchmark.
FACULTY_ID = "executive_functions/cognitive_flexibility"

# Default prefix displayed in the header of each episode.
TURN_HEADER_PREFIX = "CogFlex Suite Task. Episode "

# Reference row counts for the tracked dataset variants bundled in the repository.
EPISODE_COUNT_BY_SPLIT = {
    "test": 10,
    "public": 120,
    "private": 96,
}

# Default path to the public dataset in the Kaggle environment.
DATASET_ROOT = Path("/kaggle/input/datasets/raptorengineer/cogflex-suite-runtime")
# Default path to the private dataset in the Kaggle environment.
PRIVATE_DATASET_ROOT = Path("/kaggle/input/datasets/raptorengineer/cogflex-suite-runtime-private")

def _rows_path_for_split(eval_split: str) -> Path:
    """Return the leaderboard rows path for the selected split."""

    if eval_split == "public":
        return DATASET_ROOT / "public_leaderboard_rows.json"
    if eval_split == "private":
        return PRIVATE_DATASET_ROOT / "private_leaderboard_rows.json"
    raise ValueError(f"unsupported eval split: {eval_split!r}")


# Notebook runtime va
EVAL_SPLIT = "private"
EXPECTED_PUBLIC_EPISODE_COUNT = EPISODE_COUNT_BY_SPLIT[EVAL_SPLIT]

# Compatibility alias used by the runtime and tests.
ROWS_PATH = _rows_path_for_split(EVAL_SPLIT)

# Matches one rendered case line, including the final label token.
LINE_RE = re.compile(r"^(?P<index>\d+)\.\s+(?P<body>.+?)\s+->\s+(?P<label>[a-z0-9_:-]+|\?)$")

# Extracts the point coordinates embedded inside each rendered case body.
POINT_RE = re.compile(r"^r1=(?P<r1>[+-]\d+),\s*r2=(?P<r2>[+-]\d+)$")


# Runtime Error Type

This section defines the notebook-specific runtime error used for structural or precondition failures during episode execution.

Using a dedicated runtime error keeps these execution-time guardrail failures separate from dataset validation errors, which makes the later task logic easier to read and debug.

In [ ]:
class KaggleExecutionError(RuntimeError):
    pass


# Response Normalization

The benchmark expects each final answer to follow a per-episode `response_spec` with the `ordered_labels` format.
The current contract uses `schema_version = "cogflex.v2"`, and the notebook rebuilds a local `output_schema` before normalizing responses for scoring. The runtime accepts structured ordered-label responses only and validates them after extraction.

String responses are parsed as JSON after optional markdown code fences are removed. Unstructured text outputs are rejected as protocol failures.

This section centralizes three responsibilities:

- validate the declared response schema
- extract model outputs from supported response shapes (ordered-label dicts, JSON strings, and objects exposing `ordered_labels`)
- convert valid outputs into a canonical ordered tuple for scoring

These helpers are shared by both data validation and task execution so that the same response contract is enforced throughout the notebook.


In [ ]:
def build_strict_output_schema(probe_count: int, label_vocab: list[str], suite_task_id: str) -> dict[str, object]:
    """Build the strict structured-output schema for the final response.

    Args:
        probe_count: Number of probe answers expected.
        label_vocab: Allowed output labels.
        suite_task_id: Suite task identifier used to name the schema.

    Returns:
        A strict JSON-schema payload for the final model response.

    """
    if not isinstance(probe_count, int) or probe_count <= 0:
        raise ValueError("response_spec.probe_count must be positive")
    if not isinstance(label_vocab, list) or len(label_vocab) < 2:
        raise ValueError("response_spec.label_vocab must contain at least two labels")
    if not isinstance(suite_task_id, str) or not suite_task_id.strip():
        raise ValueError("response_spec.suite_task_id must be a non-empty string")
    schema_id = re.sub(r"[^a-z0-9]+", "_", suite_task_id.lower()).strip("_") or "cogflex"
    return {
        "type": "object",
        "title": f"cogflex_{schema_id}_response",
        "additionalProperties": False,
        "required": ["ordered_labels"],
        "properties": {
            "ordered_labels": {
                "type": "array",
                "items": {"type": "string", "enum": list(label_vocab)},
                "minItems": probe_count,
                "maxItems": probe_count,
            }
        },
    }


def _normalize_response_spec(spec: object) -> dict[str, object]:
    """Normalize and validate a response specification.

    Args:
        spec: Candidate response specification payload.

    Returns:
        A normalized ordered-labels response specification with an
        explicit strict output schema.

    """
    if not isinstance(spec, dict):
        raise ValueError("response_spec must be a JSON object")
    if spec.get("format") != "ordered_labels":
        raise ValueError("unsupported response format")
    probe_count = spec.get("probe_count")
    label_vocab = spec.get("label_vocab")
    if not isinstance(probe_count, int) or probe_count <= 0:
        raise ValueError("response_spec.probe_count must be positive")
    if not isinstance(label_vocab, list) or len(label_vocab) < 2:
        raise ValueError("response_spec.label_vocab must contain at least two labels")
    normalized = [str(label).strip() for label in label_vocab]
    if len(set(normalized)) != len(normalized):
        raise ValueError("response_spec.label_vocab must be unique")
    suite_task_id = str(spec.get("suite_task_id") or TASK_NAME).strip()
    if not suite_task_id:
        raise ValueError("response_spec.suite_task_id must be a non-empty string")
    if "output_schema" in spec and not isinstance(spec["output_schema"], dict):
        raise ValueError("response_spec.output_schema must be a JSON object when provided")
    output_schema = build_strict_output_schema(probe_count, normalized, suite_task_id)
    return {
        "schema_version": str(spec.get("schema_version") or "cogflex.v2"),
        "format": "ordered_labels",
        "probe_count": probe_count,
        "label_vocab": normalized,
        "suite_task_id": suite_task_id,
        "output_schema": output_schema,
    }


def _strip_structured_response_fences(response: str) -> str:
    """Remove optional markdown fences around a structured JSON response."""

    trimmed = response.strip()
    if trimmed.startswith(("```json", "```")):
        trimmed = re.sub(r"^```(?:json)?\s*", "", trimmed)
        trimmed = re.sub(r"\s*```\s*$", "", trimmed)
    return trimmed.strip()


def _extract_sequence_response(response: object, probe_count: int) -> tuple[str, ...] | None:
    """Extract an ordered probe response from supported response shapes.

    Handles structured dicts, JSON-encoded strings (with optional
    markdown fences), and objects exposing an `ordered_labels` field.

    Args:
        response: Raw model response to inspect.
        probe_count: Expected number of probe outputs.

    Returns:
        A tuple of extracted labels, or `None` when the response shape is
        unsupported or incomplete.

    """
    if isinstance(response, str):
        trimmed = _strip_structured_response_fences(response)
        if trimmed.startswith(("{", "[")):
            try:
                parsed = json.loads(trimmed)
                return _extract_sequence_response(parsed, probe_count)
            except (json.JSONDecodeError, ValueError):
                pass
        return None
    if isinstance(response, (list, tuple)):
        return tuple(str(item).strip() for item in response)
    if isinstance(response, dict):
        if "ordered_labels" in response:
            return _extract_sequence_response(response["ordered_labels"], probe_count)
        return None
    if is_dataclass(response) and hasattr(response, "ordered_labels"):
        return _extract_sequence_response(getattr(response, "ordered_labels"), probe_count)
    if hasattr(response, "ordered_labels"):
        return _extract_sequence_response(getattr(response, "ordered_labels"), probe_count)
    return None


def _parse_probe_response(response: object, response_spec: object) -> dict[str, object]:
    """Parse a final model response and classify any protocol failures.

    Args:
        response: Raw model response to inspect.
        response_spec: Declared response specification for the episode.

    Returns:
        A payload containing normalized predictions when available and a
        score status describing protocol compatibility.

    """
    normalized_spec = _normalize_response_spec(response_spec)
    values = _extract_sequence_response(response, normalized_spec["probe_count"])
    if values is None:
        return {"score_status": "schema_format_failure", "predictions": None}
    if len(values) != normalized_spec["probe_count"]:
        return {"score_status": "wrong_label_count", "predictions": None}
    if any(value not in normalized_spec["label_vocab"] for value in values):
        return {"score_status": "invalid_label_vocab", "predictions": None}
    return {"score_status": "ok", "predictions": values}


def normalize_ordered_labels(response: object, response_spec: object) -> tuple[str, ...] | None:
    """Normalize a model response into ordered labels for scoring.

    Args:
        response: Raw model response to normalize.
        response_spec: Declared response specification for the episode.

    Returns:
        A normalized tuple of ordered labels, or `None` when the response
        cannot be scored.

    """
    parsed = _parse_probe_response(response, response_spec)
    if parsed["score_status"] == "schema_format_failure" or parsed["score_status"] == "wrong_label_count":
        return None
    if parsed["score_status"] == "invalid_label_vocab":
        raise ValueError("response contains a label outside label_vocab")
    return parsed["predictions"]


# Parsing Helpers

This section extracts structured case items from the textual turn payloads used by each episode.

The helpers here parse individual case lines, filter the visible items for evidence or decision turns, and build the response instruction string that later validation expects to find in the final prompt.

No dataset files are loaded here. This section only turns raw markdown-like text into reusable structured helpers for the later validation and loading steps.

In [ ]:
def _parse_case_line(line: str) -> dict[str, object] | None:
    """Parse one rendered case line into a structured item.

    Args:
        line: Rendered line from an evidence or decision turn.

    Returns:
        The parsed item payload, or `None` when the line is not a case row.

    """
    match = LINE_RE.match(line.strip())
    if match is None:
        return None
    payload: dict[str, object] = {"index": int(match.group("index")), "label": match.group("label")}
    point_found = False
    for chunk in (part.strip() for part in match.group("body").split("|")):
        point_match = POINT_RE.match(chunk)
        if point_match is not None:
            payload["r1"] = int(point_match.group("r1"))
            payload["r2"] = int(point_match.group("r2"))
            point_found = True
            continue
        key, value = chunk.split("=", 1)
        payload[key.strip()] = value.strip()
    if not point_found:
        raise ValueError(f"missing coordinates in line: {line!r}")
    return payload


def _parse_turn_items(turn: str, kind: str) -> list[dict[str, object]]:
    """Parse the visible items from a rendered turn.

    Args:
        turn: Rendered turn text.
        kind: Turn kind used to filter visible labels.

    Returns:
        Parsed item payloads for the requested turn kind.

    """
    items = []
    for line in turn.splitlines():
        parsed = _parse_case_line(line)
        if parsed is None:
            continue
        if kind == "evidence" and parsed["label"] == "?":
            continue
        if kind == "decision" and parsed["label"] != "?":
            continue
        items.append(parsed)
    return items


def _response_instruction(spec: dict[str, object]) -> str:
    """Render the decision-turn response instruction.

    Args:
        spec: Response specification payload.

    Returns:
        The instruction string expected in the final decision turn.

    """
    vocab = ", ".join(str(label) for label in spec["label_vocab"])
    return f"Return exactly {spec['probe_count']} outputs in order, one per probe. Use only labels from: {vocab}."


def _final_response_instruction(spec: dict[str, object]) -> str:
    """Render the stricter provider-agnostic scoring instruction.

    Args:
        spec: Response specification payload.

    Returns:
        A compact JSON-only instruction for the final scored prompt.

    """
    vocab = ", ".join(str(label) for label in spec["label_vocab"])
    return (
        f'Return only a JSON object of the form {{"ordered_labels":[...]}}. '
        f'"ordered_labels" must contain exactly {spec["probe_count"]} labels in probe order. '
        f'Use only labels from: {vocab}. No markdown, no code fences, no explanations, no extra keys.'
    )


# Validation Helpers

This section validates each episode against the flexible benchmark contract after the response schema and parsing helpers have already been defined.

It checks turn headers, item counts, decision-turn constraints, label-vocabulary consistency, and batch-level expectations before any rows are used by the benchmark runtime.

These helpers keep the notebook strict about malformed episodes while remaining independent from file loading and dataframe assembly.

In [ ]:
def _validate_turn(turn: str, episode_id: str, turn_index: int, total_turns: int, spec: dict[str, object], response_spec: dict[str, object]) -> None:
    """Validate one rendered turn against its declared specification.

    Args:
        turn: Rendered turn text.
        episode_id: Episode identifier used in the turn header.
        turn_index: One-based turn position.
        total_turns: Total number of turns in the episode.
        spec: Declared turn specification.
        response_spec: Normalized response specification for the episode.

    Returns:
        None.

    """
    expected_header = f"{TURN_HEADER_PREFIX}{episode_id}. Turn {turn_index} of {total_turns}."
    if not turn.startswith(expected_header):
        raise ValueError(f"episode {episode_id}: malformed header for turn {turn_index}")
    items = _parse_turn_items(turn, str(spec["kind"]))
    if len(items) != int(spec["item_count"]):
        raise ValueError(f"episode {episode_id}: turn {turn_index} count does not match turn_specs")
    if spec["kind"] == "decision":
        if int(spec["item_count"]) != int(response_spec["probe_count"]):
            raise ValueError(f"episode {episode_id}: decision turn does not match probe_count")
        if _response_instruction(response_spec) not in turn:
            raise ValueError(f"episode {episode_id}: decision turn must include the response instruction")
    else:
        for item in items:
            if str(item["label"]) not in response_spec["label_vocab"]:
                raise ValueError(f"episode {episode_id}: evidence turn label is outside label_vocab")


def _normalize_final_probe_targets(values: object, response_spec: dict[str, object]) -> tuple[str, ...]:
    """Normalize the gold probe targets for a validated episode.

    Args:
        values: Candidate final probe targets.
        response_spec: Normalized response specification for the episode.

    Returns:
        The normalized final probe targets.

    """
    normalized = normalize_ordered_labels(values, response_spec)
    if normalized is None:
        raise ValueError("final_probe_targets must contain exactly probe_count valid labels")
    return normalized


def _normalize_boolean_field(value: object, field_name: str) -> bool:
    """Validate and normalize a JSON boolean field.

    Args:
        value: Candidate boolean payload.
        field_name: Fully qualified field name for error reporting.

    Returns:
        The normalized boolean value.

    """
    if isinstance(value, bool):
        return value
    raise ValueError(f"{field_name} must be a boolean")


def _normalize_probe_metadata(
    values: object,
    targets: tuple[str, ...],
    probe_annotations: object | None = None,
    *,
    require_complete: bool = False,
) -> tuple[dict[str, object], ...]:
    """Normalize rich or legacy per-probe scoring metadata.

    Args:
        values: Candidate rich per-probe metadata payload.
        targets: Gold target labels aligned to the probe sequence.
        probe_annotations: Optional legacy congruency annotations.

    Returns:
        Normalized per-probe metadata.

    """
    annotations: tuple[str | None, ...]
    if probe_annotations is None:
        annotations = tuple(None for _ in targets)
    else:
        if not isinstance(probe_annotations, (list, tuple)) or len(probe_annotations) != len(targets):
            raise ValueError("probe_annotations must align with final_probe_targets")
        annotations = tuple(str(value).strip() for value in probe_annotations)
        if any(value not in {"congruent", "incongruent"} for value in annotations):
            raise ValueError("probe_annotations values must be congruent or incongruent")
    if values is None:
        return tuple(
            {
                "probe_index": index,
                "target_label": target,
                "obsolete_rule_label": None,
                "congruency": annotations[index - 1] or "congruent",
                "requires_switch": annotations[index - 1] == "incongruent",
            }
            for index, target in enumerate(targets, start=1)
        )
    if not isinstance(values, (list, tuple)) or len(values) != len(targets):
        raise ValueError("probe_metadata must align with final_probe_targets")
    normalized: list[dict[str, object]] = []
    for index, (target, payload, fallback_congruency) in enumerate(zip(targets, values, annotations), start=1):
        if not isinstance(payload, dict):
            raise ValueError("probe_metadata entries must be JSON objects")
        if require_complete and "target_label" not in payload:
            raise ValueError("probe_metadata.target_label is required")
        target_label = str(payload["target_label"] if require_complete else payload.get("target_label") or target).strip()
        if target_label != target:
            raise ValueError("probe_metadata.target_label must match final_probe_targets")
        if require_complete and "obsolete_rule_label" not in payload:
            raise ValueError("probe_metadata.obsolete_rule_label is required")
        obsolete_raw = payload["obsolete_rule_label"] if require_complete else payload.get("obsolete_rule_label")
        obsolete_rule_label = None if obsolete_raw is None or str(obsolete_raw).strip() == "" else str(obsolete_raw).strip()
        if require_complete and "congruency" not in payload:
            raise ValueError("probe_metadata.congruency is required")
        congruency = str(
            payload["congruency"]
            if require_complete
            else payload.get("congruency") or fallback_congruency or ("incongruent" if obsolete_rule_label is not None and obsolete_rule_label != target else "congruent")
        ).strip()
        if congruency not in {"congruent", "incongruent"}:
            raise ValueError("probe_metadata.congruency must be congruent or incongruent")
        if fallback_congruency is not None and congruency != fallback_congruency:
            raise ValueError("probe_metadata.congruency must match probe_annotations")
        route_metadata = payload.get("route_metadata")
        if route_metadata is not None and not isinstance(route_metadata, dict):
            raise ValueError("probe_metadata.route_metadata must be a JSON object when provided")
        if require_complete and "requires_switch" not in payload:
            raise ValueError("probe_metadata.requires_switch is required")
        if require_complete or "requires_switch" in payload:
            requires_switch = _normalize_boolean_field(payload["requires_switch"], "probe_metadata.requires_switch")
        else:
            requires_switch = False
        normalized_payload = {
            "probe_index": index,
            "target_label": target_label,
            "obsolete_rule_label": obsolete_rule_label,
            "congruency": congruency,
            "requires_switch": requires_switch,
        }
        if route_metadata:
            normalized_payload["route_metadata"] = {str(key): str(value) for key, value in route_metadata.items()}
        if "active_rule_id" in payload and payload.get("active_rule_id") is not None:
            normalized_payload["active_rule_id"] = str(payload["active_rule_id"])
        normalized.append(normalized_payload)
    return tuple(normalized)


def _validate_row(row: dict[str, object]) -> None:
    """Validate one benchmark row before evaluation.

    Args:
        row: Benchmark row payload to validate.

    Returns:
        None.

    """
    episode_id = str(row["episode_id"])
    analysis = row["analysis"]
    inference = row["inference"]
    response_spec = _normalize_response_spec(inference["response_spec"])
    if analysis["faculty_id"] != FACULTY_ID:
        raise ValueError(f"episode {episode_id}: unsupported faculty_id")
    if analysis["difficulty_bin"] not in {"hard", "medium"}:
        raise ValueError(f"episode {episode_id}: unsupported difficulty_bin")
    turns = inference["turns"]
    specs = inference["turn_specs"]
    if not isinstance(turns, list) or not isinstance(specs, list) or len(turns) != len(specs) or len(turns) < 2:
        raise ValueError(f"episode {episode_id}: invalid flexible turn layout")
    if [spec["kind"] for spec in specs].count("decision") != 1 or specs[-1]["kind"] != "decision":
        raise ValueError(f"episode {episode_id}: expected a final decision turn")
    for turn_index, (turn, spec) in enumerate(zip(turns, specs), start=1):
        _validate_turn(turn, episode_id, turn_index, len(turns), spec, response_spec)
    scoring = row.get("scoring")
    if EVAL_SPLIT == "public":
        if not isinstance(scoring, dict):
            raise ValueError(f"episode {episode_id}: public rows must include scoring")
        if "final_probe_targets" not in scoring:
            raise ValueError(f"episode {episode_id}: scoring must include final_probe_targets")
        if "probe_annotations" not in scoring:
            raise ValueError(f"episode {episode_id}: scoring must include probe_annotations")
        targets = _normalize_final_probe_targets(scoring["final_probe_targets"], response_spec)
        _normalize_probe_metadata(scoring.get("probe_metadata"), targets, scoring.get("probe_annotations"))
    elif "scoring" in row:
        raise ValueError(f"episode {episode_id}: private rows must not include scoring")


def _validate_batch(rows: list[dict[str, object]]) -> None:
    """Validate a batch of benchmark rows for the active split.

    Args:
        rows: Benchmark rows to validate.

    Returns:
        None.

    """
    if not rows:
        raise RuntimeError("benchmark dataset must contain at least one row")
    if EVAL_SPLIT == "public" and EXPECTED_PUBLIC_EPISODE_COUNT > 0 and len(rows) != EXPECTED_PUBLIC_EPISODE_COUNT:
        raise RuntimeError(f"expected {EXPECTED_PUBLIC_EPISODE_COUNT} public rows, found {len(rows)}")
    for row in rows:
        _validate_row(row)



# Data Loading And Assembly

This section turns the configured dataset paths into validated benchmark inputs after parsing and validation helpers are already in place.

It reads the selected rows, attaches hidden scoring targets when the private split is active, and materializes the dataframe view consumed by the benchmark runtime.

## Outputs

After this section, the notebook has:

- `leaderboard_rows`: rows prepared for evaluation
- `scored_rows`: rows with targets attached for scoring
- `df`: a dataframe view used by the benchmark runtime

In [ ]:
def _load_rows(path: Path) -> list[dict[str, object]]:
    """Load and validate benchmark rows from disk.

    Args:
        path: JSON file containing benchmark rows.

    Returns:
        The validated benchmark rows.

    """
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, list):
        raise RuntimeError(f"expected a JSON list at {path}")
    _validate_batch(payload)
    return payload


def _load_private_answer_key(path: Path) -> dict[str, object]:
    """Load the external private answer key.

    Args:
        path: JSON file containing the private answer key.

    Returns:
        The parsed private answer-key payload.

    """
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError("private answer key must be a JSON object")
    if payload.get("split") != "private":
        raise RuntimeError("private answer key must declare split='private'")
    episodes = payload.get("episodes")
    if not isinstance(episodes, list):
        raise RuntimeError("private answer key must expose an episodes list")
    return payload


def _resolve_private_answer_key_path() -> Path | None:
    """Resolve the bundled private answer-key path for the active runtime configuration."""

    if EVAL_SPLIT != "private":
        return None
    default_private_answer_key_path = PRIVATE_DATASET_ROOT / "private_answer_key.json"
    return default_private_answer_key_path if default_private_answer_key_path.exists() else None


def _scoring_payload_from_targets(
    targets: tuple[str, ...],
    probe_metadata: tuple[dict[str, object], ...],
) -> dict[str, object]:
    """Build the normalized scoring payload attached to a private row."""

    return {
        "final_probe_targets": list(targets),
        "probe_metadata": list(probe_metadata),
        "probe_annotations": [str(item["congruency"]) for item in probe_metadata],
    }


def _attached_private_row(row: dict[str, object], scoring: dict[str, object]) -> dict[str, object]:
    """Copy a private row and attach its normalized scoring payload."""

    return {
        "episode_id": row["episode_id"],
        "analysis": dict(row["analysis"]),
        "inference": dict(row["inference"]),
        "scoring": scoring,
    }


def _attach_private_scoring(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    """Attach hidden scoring targets to private rows.

    Args:
        rows: Private benchmark rows without scoring payloads.

    Returns:
        Copies of the rows with final probe targets attached.

    """
    private_answer_key_path = _resolve_private_answer_key_path()
    if private_answer_key_path is None:
        raise RuntimeError("Private split requires bundled private_answer_key.json")
    answer_key = _load_private_answer_key(private_answer_key_path)
    row_episode_ids = {str(row["episode_id"]) for row in rows}
    episodes: dict[str, dict[str, object]] = {}
    for item in answer_key["episodes"]:
        if not isinstance(item, dict):
            raise RuntimeError("private answer key episodes must be JSON objects")
        episode_id = str(item.get("episode_id", "")).strip()
        if not episode_id:
            raise RuntimeError("private answer key episode_id must be non-empty")
        if episode_id in episodes:
            raise RuntimeError(f"private answer key duplicates episode_id {episode_id}")
        episodes[episode_id] = item
    answer_key_episode_ids = set(episodes)
    missing_episode_ids = sorted(row_episode_ids - answer_key_episode_ids)
    extra_episode_ids = sorted(answer_key_episode_ids - row_episode_ids)
    if missing_episode_ids or extra_episode_ids:
        raise RuntimeError(
            "private answer key episode set mismatch: "
            f"missing={missing_episode_ids}, extra={extra_episode_ids}"
        )
    attached = []
    for row in rows:
        episode = episodes.get(str(row["episode_id"]))
        if episode is None:
            raise RuntimeError(f"missing answer key episode {row['episode_id']}")
        if episode.get("inference") != row["inference"]:
            raise RuntimeError(f"answer key inference mismatch for episode {row['episode_id']}")
        response_spec = _normalize_response_spec(row["inference"]["response_spec"])
        targets = _normalize_final_probe_targets(episode.get("final_probe_targets"), response_spec)
        probe_metadata = _normalize_probe_metadata(
            episode.get("probe_metadata"),
            targets,
            episode.get("probe_annotations"),
            require_complete=True,
        )
        scoring = _scoring_payload_from_targets(targets, probe_metadata)
        attached.append(_attached_private_row(row, scoring))
    return attached


def load_selected_rows() -> list[dict[str, object]]:
    """Load the rows configured for the active evaluation split.

    Returns:
        The validated benchmark rows selected for evaluation.

    """
    return _load_rows(_rows_path_for_split(EVAL_SPLIT))


def attach_selected_scoring(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    """Attach scoring information for the active evaluation split.

    Args:
        rows: Benchmark rows to prepare for scoring.

    Returns:
        Public rows unchanged, or private rows with hidden targets attached.

    """
    if EVAL_SPLIT == "public":
        return rows
    return _attach_private_scoring(rows)



def _evaluation_row(row: dict[str, object]) -> dict[str, object]:
    """Build one evaluation row consumed by `run_flexible_task.evaluate(...)`."""

    return {
        "turns": row["inference"]["turns"],
        "response_spec": row["inference"]["response_spec"],
        "final_probe_targets": row["scoring"]["final_probe_targets"],
        "probe_metadata": row["scoring"].get("probe_metadata"),
        "probe_annotations": row["scoring"].get("probe_annotations"),
    }


# Raw rows selected for the active split before any scoring attachment.
leaderboard_rows = load_selected_rows()

# Evaluation rows enriched with scoring fields when the private split is active.
scored_rows = attach_selected_scoring(leaderboard_rows)

# Flat dataframe view consumed by `run_flexible_task.evaluate(...)`.
df = pd.DataFrame([_evaluation_row(row) for row in scored_rows])

# Episode Execution And Scoring

This section defines how a single validated episode is run against a model and how the final answer is scored.

The execution path sends each evidence turn with its own `llm.prompt(...)` call, then submits the final decision turn as the only scored prompt, normalizes the model response, and compares the ordered labels against the expected probe targets.

No evaluation runs here yet. This section only defines the execution primitive that the registered Kaggle task will call later.


In [ ]:
def score_episode(
    predictions: tuple[str, ...] | None,
    targets: tuple[str, ...],
    probe_metadata: tuple[dict[str, object], ...] | None = None,
    probe_annotations: tuple[str, ...] | None = None,
    *,
    score_status: str = "ok",
) -> dict:
    """Score one episode from ordered predictions and targets.

    Args:
        predictions: Normalized model predictions, or `None` when missing.
        targets: Gold probe labels for the episode.
        probe_metadata: Optional rich per-probe scoring metadata.
        probe_annotations: Optional legacy congruency annotations.
        score_status: Protocol or scoring status for the episode result.

    Returns:
        A scoring payload with numerator, denominator, stored predictions,
        scoring status, and richer switching diagnostics.

    """
    normalized_probe_metadata = _normalize_probe_metadata(probe_metadata, targets, probe_annotations)
    if predictions is None:
        predictions = tuple("" for _ in targets)
        numerator = 0
    else:
        numerator = sum(pred == target for pred, target in zip(predictions, targets))
    if score_status == "ok":
        score_status = "correct" if numerator == len(targets) else "cognitive_mismatch"

    # Aggregate counters used to derive the episode-level diagnostic metrics.
    inc_num = inc_den = cong_num = cong_den = 0
    first_probe_num = first_probe_den = 0
    obsolete_num = obsolete_den = 0
    switch_num = switch_den = 0

    # Tracks whether the previous probe already belonged to an active switch block.
    previous_requires_switch = False
    for index, (pred, target, metadata) in enumerate(zip(predictions, targets, normalized_probe_metadata), start=1):
        congruency = str(metadata["congruency"])
        if congruency == "incongruent":
            inc_den += 1
            if pred == target:
                inc_num += 1
        else:
            cong_den += 1
            if pred == target:
                cong_num += 1
        requires_switch = bool(metadata["requires_switch"])
        if requires_switch and not previous_requires_switch:
            first_probe_den += 1
            if pred == target:
                first_probe_num += 1
        obsolete_label = metadata["obsolete_rule_label"]
        if obsolete_label is not None and str(obsolete_label).strip() != "":
            obsolete_den += 1
            if pred == obsolete_label:
                obsolete_num += 1
        if requires_switch:
            switch_den += 1
            if pred == target:
                switch_num += 1
        previous_requires_switch = requires_switch
    result = {
        "numerator": numerator,
        "denominator": len(targets),
        "predictions": list(predictions),
        "score_status": score_status,
        "incongruent_numerator": inc_num,
        "incongruent_denominator": inc_den,
        "congruent_numerator": cong_num,
        "congruent_denominator": cong_den,
        "first_probe_numerator": first_probe_num,
        "first_probe_denominator": first_probe_den,
        "obsolete_rule_error_numerator": obsolete_num,
        "obsolete_rule_error_denominator": obsolete_den,
        "requires_switch_numerator": switch_num,
        "requires_switch_denominator": switch_den,
        "scorable": score_status in ("correct", "cognitive_mismatch"),
    }
    return result


def _render_episode_prompt(final_turn: str, response_spec: dict[str, object]) -> str:
    """Render the final scored prompt for one episode.

    Args:
        final_turn: Rendered final decision turn for the episode.
        response_spec: Normalized response contract for the final decision turn.

    Returns:
        The scored final decision turn with the strict response contract.

    """
    return f"{final_turn}\n\n{_final_response_instruction(response_spec)}"


@kbench.task(store_task=False)
def run_flexible_task(
    llm: object,
    turns: list[str],
    response_spec: dict[str, object],
    final_probe_targets: tuple[str, ...],
    probe_metadata: tuple[dict[str, object], ...] | None = None,
    probe_annotations: tuple[str, ...] | None = None,
) -> dict:
    """Execute one flexible episode against an LLM and score it.

    Args:
        llm: Model interface exposing `prompt`.
        turns: Ordered rendered turns for the episode.
        response_spec: Response contract for the final decision turn.
        final_probe_targets: Gold probe labels for scoring.
        probe_metadata: Optional rich per-probe scoring metadata.
        probe_annotations: Optional legacy per-probe congruency annotations.

    Returns:
        The episode scoring payload.

    """
    if len(turns) < 2:
        raise KaggleExecutionError("expected at least two turns")
    normalized_spec = _normalize_response_spec(response_spec)
    for turn in turns[:-1]:
        try:
            llm.prompt(turn)
        except Exception as exc:
            print(f"Evidence prompt failed: {exc!r}", file=sys.stderr)
            return score_episode(None, final_probe_targets, probe_metadata, probe_annotations, score_status="prompt_failure")
    final_prompt = _render_episode_prompt(turns[-1], normalized_spec)
    try:
        response = llm.prompt(final_prompt)
    except Exception as exc:
        print(f"Final prompt failed: {exc!r}", file=sys.stderr)
        return score_episode(None, final_probe_targets, probe_metadata, probe_annotations, score_status="prompt_failure")
    parsed = _parse_probe_response(response, response_spec)
    return score_episode(
        parsed["predictions"],
        final_probe_targets,
        probe_metadata,
        probe_annotations,
        score_status=str(parsed["score_status"]),
    )


# Benchmark Summary And Task Registration

This section aggregates per-episode results and registers the Kaggle task entry point `cogflex_suite_flexible`.

The benchmark summary reports:

- overall micro accuracy
- macro accuracy across suite tasks
- breakdowns by suite task
- breakdowns by structure family
- breakdowns by difficulty bin

After this section runs, the notebook has a fully registered task ready to be executed and then selected by `%choose`.

In [ ]:
def _build_compact_suite_summary(full_summary: dict[str, object]) -> dict[str, object]:
    """Extract the compact default summary from the full diagnostic summary.

    Args:
        full_summary: The full summary with all diagnostic fields.

    Returns:
        A summary with only the 8 essential metric keys.

    """
    return {
        "score": full_summary["score"],
        "protocol_valid_rate": full_summary["protocol_valid_rate"],
        "scorable_episodes": full_summary["scorable_episodes"],
        "episodes": full_summary["episodes"],
        "macro_accuracy": full_summary["macro_accuracy"],
        "incongruent_accuracy": full_summary["incongruent_accuracy"],
        "first_probe_accuracy": full_summary["first_probe_accuracy"],
        "obsolete_rule_error_rate": full_summary["obsolete_rule_error_rate"],
    }


def summarize_suite_benchmark(runs, rows: list[dict[str, object]], *, include_debug: bool = False) -> dict[str, object]:
    """Aggregate benchmark metrics across all evaluated episodes.

    Args:
        runs: Completed Kaggle benchmark run collection.
        rows: Scored benchmark rows aligned with the run outputs.
        include_debug: If True, return the full diagnostic summary with all
            detailed fields. If False (default), return only the 8 essential
            metrics.

    Returns:
        Aggregate macro, micro, switching, perseverative, and slice metrics
        for the benchmark.

    """
    if not runs:
        raise RuntimeError("evaluation produced no successful runs")
    results_df = runs.as_dataframe().reset_index(drop=True)
    if len(results_df) != len(rows):
        raise RuntimeError(f"evaluation completed {len(results_df)} of {len(rows)} benchmark episodes")

    def _safe_div(numerator: int, denominator: int) -> float:
        return numerator / denominator if denominator > 0 else 0.0

    def _metrics_from_counts(counts: dict[str, int]) -> dict[str, float]:
        micro_accuracy = _safe_div(counts["numerator"], counts["denominator"])
        incongruent_accuracy = _safe_div(counts["inc_num"], counts["inc_den"])
        congruent_accuracy = _safe_div(counts["cong_num"], counts["cong_den"])
        first_probe_accuracy = _safe_div(counts["first_num"], counts["first_den"])
        obsolete_rule_error_rate = _safe_div(counts["obsolete_num"], counts["obsolete_den"])
        requires_switch_accuracy = _safe_div(counts["switch_num"], counts["switch_den"])
        return {
            "micro_accuracy": micro_accuracy,
            "incongruent_accuracy": incongruent_accuracy,
            "congruent_accuracy": congruent_accuracy,
            "first_probe_accuracy": first_probe_accuracy,
            "obsolete_rule_error_rate": obsolete_rule_error_rate,
            "requires_switch_accuracy": requires_switch_accuracy,
            "switch_cost": congruent_accuracy - incongruent_accuracy,
        }

    # Running totals across the full benchmark.
    overall_counts = {
        "numerator": 0,
        "denominator": 0,
        "inc_num": 0,
        "inc_den": 0,
        "cong_num": 0,
        "cong_den": 0,
        "first_num": 0,
        "first_den": 0,
        "obsolete_num": 0,
        "obsolete_den": 0,
        "switch_num": 0,
        "switch_den": 0,
    }

    # Bucketed counters used for macro and slice summaries.
    per_task: dict[str, dict[str, int]] = {}
    per_structure: dict[str, dict[str, int]] = {}
    per_difficulty: dict[str, dict[str, int]] = {}

    # Counts episodes that produced a protocol-valid, scoreable result.
    scorable_count = 0
    for row, result in zip(rows, results_df.result):
        suite_task_id = row["analysis"]["suite_task_id"]
        structure_family_id = row["analysis"]["structure_family_id"]
        difficulty_bin = row["analysis"]["difficulty_bin"]

        # Episode-local counts normalized to integers before aggregation.
        counts = {
            "numerator": int(result["numerator"]),
            "denominator": int(result["denominator"]),
            "inc_num": int(result.get("incongruent_numerator", 0)),
            "inc_den": int(result.get("incongruent_denominator", 0)),
            "cong_num": int(result.get("congruent_numerator", 0)),
            "cong_den": int(result.get("congruent_denominator", 0)),
            "first_num": int(result.get("first_probe_numerator", 0)),
            "first_den": int(result.get("first_probe_denominator", 0)),
            "obsolete_num": int(result.get("obsolete_rule_error_numerator", 0)),
            "obsolete_den": int(result.get("obsolete_rule_error_denominator", 0)),
            "switch_num": int(result.get("requires_switch_numerator", 0)),
            "switch_den": int(result.get("requires_switch_denominator", 0)),
        }
        scorable = result.get("scorable", True)
        if scorable:
            scorable_count += 1
        else:
            # Keep protocol validity separate from obsolete-rule cognition metrics.
            counts["obsolete_num"] = 0
            counts["obsolete_den"] = 0
        for key, value in counts.items():
            overall_counts[key] += value
        for bucket, key in ((per_task, suite_task_id), (per_structure, structure_family_id), (per_difficulty, difficulty_bin)):
            entry = bucket.setdefault(key, {name: 0 for name in overall_counts})
            for name, value in counts.items():
                entry[name] += value
    per_task_accuracy = {key: _safe_div(value["numerator"], value["denominator"]) for key, value in sorted(per_task.items())}
    macro_accuracy = sum(per_task_accuracy.values()) / len(per_task_accuracy)
    overall_metrics = _metrics_from_counts(overall_counts)
    protocol_valid_rate = scorable_count / len(rows) if len(rows) > 0 else 0.0

    # Composite benchmark score used by the registered Kaggle task.
    score = (
        macro_accuracy
        + overall_metrics["incongruent_accuracy"]
        + overall_metrics["first_probe_accuracy"]
        + protocol_valid_rate * (1.0 - overall_metrics["obsolete_rule_error_rate"])
    ) / 4.0
    full_summary = {
        "score": score,
        "score_formula": "average(macro_accuracy, incongruent_accuracy, first_probe_accuracy, protocol_valid_rate * (1 - obsolete_rule_error_rate))",
        "macro_accuracy": macro_accuracy,
        "micro_accuracy": overall_metrics["micro_accuracy"],
        "incongruent_accuracy": overall_metrics["incongruent_accuracy"],
        "congruent_accuracy": overall_metrics["congruent_accuracy"],
        "first_probe_accuracy": overall_metrics["first_probe_accuracy"],
        "obsolete_rule_error_rate": overall_metrics["obsolete_rule_error_rate"],
        "requires_switch_accuracy": overall_metrics["requires_switch_accuracy"],
        "switch_cost": overall_metrics["switch_cost"],
        "numerator": overall_counts["numerator"],
        "denominator": overall_counts["denominator"],
        "incongruent_numerator": overall_counts["inc_num"],
        "incongruent_denominator": overall_counts["inc_den"],
        "congruent_numerator": overall_counts["cong_num"],
        "congruent_denominator": overall_counts["cong_den"],
        "first_probe_numerator": overall_counts["first_num"],
        "first_probe_denominator": overall_counts["first_den"],
        "obsolete_rule_error_numerator": overall_counts["obsolete_num"],
        "obsolete_rule_error_denominator": overall_counts["obsolete_den"],
        "requires_switch_numerator": overall_counts["switch_num"],
        "requires_switch_denominator": overall_counts["switch_den"],
        "episodes": len(rows),
        "scorable_episodes": scorable_count,
        "protocol_valid_rate": protocol_valid_rate,
        "per_task_accuracy": per_task_accuracy,
        "per_task_metrics": {key: _metrics_from_counts(value) for key, value in sorted(per_task.items())},
        "structure_family_accuracy": {key: _safe_div(value["numerator"], value["denominator"]) for key, value in sorted(per_structure.items())},
        "difficulty_bin_accuracy": {key: _safe_div(value["numerator"], value["denominator"]) for key, value in sorted(per_difficulty.items())},
    }
    if include_debug:
        return full_summary
    return _build_compact_suite_summary(full_summary)


@kbench.task(name="cogflex_suite_flexible", description="Cognitive flexibility benchmark within executive functions with variable turn structures and label vocabularies.")
def cogflex_suite_flexible(llm) -> float:
    """Run the full CogFlex flexible benchmark for one model.

    Args:
        llm: Model interface to evaluate.

    Returns:
        The benchmark score reported to Kaggle.

    """
    runs = run_flexible_task.evaluate(llm=[llm], evaluation_data=df)
    summary = summarize_suite_benchmark(runs, scored_rows, include_debug=True)
    print(json.dumps(summary, indent=2))
    return summary["score"]

# Run The Main Benchmark Task

Run the top-level benchmark task once so Kaggle records the latest run file for `cogflex_suite_flexible`.

This should call the main task, not the per-episode helper, because the helper expects row fields like `turns` and `response_spec` that are supplied through `.evaluate(...)`.

In [ ]:
cogflex_suite_flexible.run(kbench.llm)

# Final Task Selection

This final step selects `cogflex_suite_flexible` as the notebook entry point for Kaggle after the latest main-task run has been recorded.

`%choose` should remain the last executable step in the notebook, after all setup, validation, execution, and registration logic has already been defined.

In [ ]:
%choose cogflex_suite_flexible